# AI Agent Demo

Minimal LangChain proof of concept that loads tools from the asyncroscopy MCP server and asks the model to use one or two of them. This will be useful to test our MCP server and new tools.

## Prereqs

Start the Tango stack and MCP server first:

```bash
uv run startup_scripts/run_servers.py
uv run startup_scripts/run_mcp.py --yaml configs/mcp.yaml
```

Install the optional AI extras:

```bash
uv sync --extra agent
# for local models do uv sync --extra agent --extra localagent
```

In [27]:
import os
from pprint import pprint

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_core.tools import BaseTool

### Connect to the running MCP server

In [30]:
MCP_URL = "http://127.0.0.1:8000/mcp"

client = MultiServerMCPClient(
    {
        "asyncroscopy": {
            "url": MCP_URL,
            "transport": "streamable_http",
        }
    }
)

tools = await client.get_tools()
print(f"Loaded {len(tools)} tool(s)")
print([tool.name for tool in tools])

Loaded 52 tool(s)
['get_data_from_key', 'list_devices', 'CAMERA_State', 'CAMERA_Status', 'DATA_State', 'DATA_Status', 'DATA_configure', 'DATA_get_config', 'DATA_register_path', 'DATA_register_save_path', 'DATA_start_tiled_server', 'DATA_stop_tiled_server', 'EDS_State', 'EDS_Status', 'FLUCAM_State', 'FLUCAM_Status', 'DigitalTwin_Connect', 'DigitalTwin_Disconnect', 'DigitalTwin_State', 'DigitalTwin_Status', 'DigitalTwin_acquire_camera_image', 'DigitalTwin_acquire_flucam_image', 'DigitalTwin_acquire_scanned_data_advanced', 'DigitalTwin_acquire_scanned_image', 'DigitalTwin_acquire_spectrum', 'DigitalTwin_auto_focus', 'DigitalTwin_blank_beam', 'DigitalTwin_calibrate_screen_current', 'DigitalTwin_get_beam_tilt', 'DigitalTwin_get_defocus', 'DigitalTwin_get_diffraction_shift', 'DigitalTwin_get_fov', 'DigitalTwin_get_image_data_cached', 'DigitalTwin_get_image_shift', 'DigitalTwin_get_screen_current', 'DigitalTwin_get_stage', 'DigitalTwin_get_status', 'DigitalTwin_move_stage', 'DigitalTwin_place

In [31]:
def filter_tools(tools: list[BaseTool], tool_names: list[str]) -> list[BaseTool]:
    return [tool for tool in tools if tool.name in tool_names]

# Filter the tools so we can test a few at a time
tools = filter_tools(tools, ["list_devices", "AutoScriptMicroscope_acquire_scanned_image", "get_data_from_key"])
print(f"Filtered to {len(tools)} tool(s): {[tool.name for tool in tools]}")

Filtered to 2 tool(s): ['get_data_from_key', 'list_devices']


### Run cell below to use an OPENAI-COMPATIBLE model

In [ ]:
model = init_chat_model(
    model="YOUR MODEL NAME",
    model_provider="openai, google_genai, etc.", # Make sure to install the right provider package for the model you want to use
    api_key="YOUR API KEY", 
)

using_local_model = False

### Run cell below to use a LOCAL model

In [32]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig
from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline

print("CUDA Available:", torch.cuda.is_available())
print("Device Count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("Current Device:", torch.cuda.get_device_name(0))
    
HF_MODEL_ID = r"C:\Users\Public\Desktop\Agents\Gemma4-31B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(HF_MODEL_ID)
hf_model = AutoModelForCausalLM.from_pretrained(
    HF_MODEL_ID,
    device_map="auto",
    quantization_config=quantization_config,
)

hf_pipeline = pipeline(
    "text-generation",
    model=hf_model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    temperature=0.2,
    return_full_text=False,
)

model = ChatHuggingFace(llm=HuggingFacePipeline(pipeline=hf_pipeline))

using_local_model = True

CUDA Available: True
Device Count: 1
Current Device: NVIDIA GeForce RTX 5090


W0712 19:43:52.235000 36944 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Loading weights:   0%|          | 0/1188 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


### Initialize Agent

In [33]:
agent = create_agent(model, tools)

def final_text(result):
    message = result["messages"][-1]
    return getattr(message, "content", message)

def tool_calls(result):
    message = result["messages"][-1]
    return getattr(message, "tool_calls", message)

### Basic prompt

In [35]:
prompt = "What tools can you access?"

if not using_local_model:
    result = await agent.ainvoke({
        "messages": prompt
    })
else :
    result = agent.invoke(
        {"messages": prompt},
        config={"configurable": {"use_async": False}}
    )

print(final_text(result))
print(tool_calls(result))

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
c:\Users\dpelaia\Documents\asyncroscopy-agent-version\.venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


I do not have access to any tools, such as Google Search or external applications, by default. I can only use tools if specific definitions and endpoints are provided to me within the context of our conversation.<turn|>
[]


### Test image acquisition

In [ ]:
prompt = "Get a scanned image via HAADF and give me the full file path."

if not using_local_model:
    result = await agent.ainvoke({
        "messages": prompt
    })
else :
    result = agent.invoke(
        {"messages": prompt},
        config={"configurable": {"use_async": False}}
    )

print(final_text(result))
print(tool_calls(result))

The scanned image has been acquired via HAADF. The file path is: `stem_image_HAADF_20260706T141203062528.h5`
